# 02 — Train a detector and export its predictions

**Work package:** WP3 · **Runtime: GPU (T4)** · **Time:** roughly 45–75 min per model

Run order (one run per GPU session; DECISION_LOG D-008, D-017):

| Run | `EXPERIMENT` | `SPLIT` | Purpose |
|---|---|---|---|
| 1 | `yolo11n_640` | `roboflow` | baseline on the split as provided |
| 2 | `yolo11n_640` | `blocked` | same model on the time-blocked split → how much did leakage inflate run 1? |
| 3 | `yolo26n_640` | `blocked` | model comparison on the leakage-free split |

Training settings are identical across runs, so only the model or the split differs.

What it does:
1. downloads the dataset (Colab's disk is wiped between sessions),
2. trains from COCO-pretrained weights, saving to Google Drive — **if Colab disconnects,
   reconnect and *Run all* again: training resumes from the last checkpoint**,
3. plots training vs validation curves (over/underfitting check),
4. runs Ultralytics' own validation on the VALID split (an independent cross-check),
5. exports raw predictions on VALID and the ground truth, so all further evaluation runs
   on CPU with our own tested metrics (D-018). **The TEST split is not touched here** (D-008).

**Free-Colab tips:** *Runtime → Change runtime type → T4 GPU*. Keep this browser tab open
and visible — idle sessions get disconnected. Stop the runtime when finished
(*Runtime → Disconnect and delete runtime*) to save GPU quota.


## 1. Settings

In [ ]:
EXPERIMENT = "yolo11n_640"   # "yolo11n_640" or "yolo26n_640"
SPLIT = "roboflow"           # "roboflow" (as provided) or "blocked" (time-blocked, leakage-free)
RUN_NAME = EXPERIMENT if SPLIT == "roboflow" else f"{EXPERIMENT}_{SPLIT}"
PUSH_RESULTS = True          # push result files to the 'colab-results' branch if a GITHUB_TOKEN secret exists
FORCE_RETRAIN = False        # True = ignore a finished run on Drive and train again

## 2. Get the latest code and record the environment

In [ ]:
import os, sys, json, subprocess, datetime, platform

REPO_URL = "https://github.com/Hudson11111/MMA3001Project.git"
REPO_DIR = "/content/MMA3001Project"

if os.path.isdir(REPO_DIR):   # always run the latest main
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q main && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

import torch, ultralytics, porkqc
env_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__,
    "ultralytics": ultralytics.__version__, "porkqc": porkqc.__version__,
    "git_commit": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip(),
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(env_info, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

## 3. Google Drive (runs and weights persist here)

In [ ]:
from pathlib import Path
from google.colab import drive
from porkqc.config import load_yaml

drive.mount("/content/drive")
cfg = load_yaml("configs/dataset.yaml")
DRIVE_DIR = Path(cfg["paths"]["drive_project_dir"])
RUNS_DIR = DRIVE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("Runs folder:", RUNS_DIR)
!ls {RUNS_DIR}

## 4. Dataset (download if missing, repair `data.yaml`, check counts)

In [ ]:
from porkqc import CLASS_NAMES
from porkqc.config import check_class_names
from porkqc.data.download import download_dataset, fix_data_yaml, read_class_names
from porkqc.data.audit import summarise_dataset

rf = cfg["roboflow"]
data_root = Path(cfg["paths"]["colab_data_dir"])
if not (data_root / "data.yaml").exists():
    data_root = download_dataset(rf["workspace"], rf["project"], rf["version"], data_root,
                                 export_format=rf["export_format"])
DATA_YAML = data_root / "data.yaml"
fix_data_yaml(DATA_YAML)
check_class_names(read_class_names(DATA_YAML), CLASS_NAMES)

expected = cfg["expected"]["split_counts"]
for split, s in summarise_dataset(data_root, len(CLASS_NAMES)).items():
    assert s.n_images == expected[split], f"{split}: {s.n_images} images, expected {expected[split]}"
print("Roboflow dataset OK:", DATA_YAML)

EVAL_ROOT = data_root
if SPLIT == "blocked":
    from porkqc.data.resplit import collect_frames, assign_time_blocks, materialise_split
    b = cfg["blocked_split"]
    frames_map, files_map, unparsed = collect_frames(data_root)   # ~1-2 min: separates same-named frames
    assign = assign_time_blocks(frames_map, b["block_size"], b["fractions"], b["gap"], b["seed"])
    EVAL_ROOT = Path(b["colab_dir"])
    counts = materialise_split(data_root, EVAL_ROOT, assign, CLASS_NAMES, files=files_map)
    DATA_YAML = EVAL_ROOT / "data.yaml"
    print(f"Blocked split (block {b['block_size']}, gap {b['gap']}): images per split {counts}; "
          f"purged scenes: {sum(v is None for v in assign.values())}; unparsed names: {len(unparsed)}")
elif SPLIT != "roboflow":
    raise ValueError("SPLIT must be 'roboflow' or 'blocked'")
print("Training data:", DATA_YAML)

## 5. Train (or resume, or skip if already finished)

`train_detector` checks Drive first: a finished run (`run_meta.json`) is skipped, an
interrupted run (`weights/last.pt`) is resumed, otherwise training starts from the
pretrained weights. Ultralytics prints one progress line per epoch.

In [ ]:
from porkqc.models.detector import load_experiment, train_detector

exp = load_experiment(f"configs/experiments/{EXPERIMENT}.yaml")
exp["name"] = RUN_NAME            # run folder on Drive, e.g. yolo11n_640_blocked
exp["split"] = SPLIT
print(json.dumps(exp, indent=1))
meta = train_detector(exp, DATA_YAML, RUNS_DIR, force=FORCE_RETRAIN)
RUN_DIR = Path(meta["run_dir"])
BEST = Path(meta["best_weights"])
print("Run folder:", RUN_DIR)
print("Already finished earlier - nothing trained." if meta["skipped"]
      else f"Training call took {meta['train_seconds'] / 60:.1f} min (resumed: {meta['resumed']})")

## 6. Training curves — is it converged? is it overfitting?

Overfitting shows up as validation loss rising while training loss keeps falling;
underfitting as both still falling steeply at the last epoch (LO: noise & overfitting).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.read_csv(RUN_DIR / "results.csv")
hist.columns = [c.strip() for c in hist.columns]
loss_names = [c.split("/", 1)[1] for c in hist.columns if c.startswith("train/") and f"val/{c.split('/', 1)[1]}" in hist.columns]
metric_cols = [c for c in hist.columns if c.startswith("metrics/")]

fig, axes = plt.subplots(1, len(loss_names) + 1, figsize=(4.2 * (len(loss_names) + 1), 3.4))
for ax, name in zip(axes, loss_names):
    ax.plot(hist["epoch"], hist[f"train/{name}"], label="train")
    ax.plot(hist["epoch"], hist[f"val/{name}"], label="validation")
    ax.set_title(name); ax.set_xlabel("epoch"); ax.legend()
for c in metric_cols:
    axes[-1].plot(hist["epoch"], hist[c], label=c.replace("metrics/", ""))
axes[-1].set_title("validation metrics"); axes[-1].set_xlabel("epoch"); axes[-1].legend(fontsize=7)
fig.suptitle(RUN_NAME); fig.tight_layout()

OUT = Path("results/train") / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "training_curves.png", dpi=110)
plt.show()

best_row = hist.loc[hist["metrics/mAP50-95(B)"].idxmax()]
print(f"epochs run: {int(hist['epoch'].max())} | best epoch: {int(best_row['epoch'])} | "
      f"best val mAP50-95: {best_row['metrics/mAP50-95(B)']:.4f} | mAP50: {best_row['metrics/mAP50(B)']:.4f}")
if "time" in hist.columns:
    print(f"total training time recorded by Ultralytics: {hist['time'].max() / 60:.1f} min")

## 7. Ultralytics' own validation on VALID (independent cross-check)

In [ ]:
from porkqc.models.detector import validate_detector
from porkqc.predictions import save_json

val_metrics = validate_detector(BEST, DATA_YAML, split="val", imgsz=exp["predict"]["imgsz"],
                                batch=16, conf=0.001, iou=exp["predict"]["iou"])
val_metrics["class_names"] = list(CLASS_NAMES)
save_json(val_metrics, OUT / "ultralytics_val_metrics.json")
print(json.dumps({k: val_metrics[k] for k in ("map50", "map50_95", "precision", "recall", "speed_ms")}, indent=1))

## 8. Export predictions on VALID (and the ground truth)

Low confidence threshold (0.001) so the full precision–recall curve can be computed
later. For YOLO26 a second variant uses the NMS-free head (`nms: false`).

In [ ]:
from porkqc.models.detector import predict_images
from porkqc.predictions import export_ground_truth
from porkqc.data.audit import list_images

valid_images = list_images(EVAL_ROOT / "valid" / "images")
pred_files = []
for variant in exp["predict_variants"]:
    kwargs = {**exp["predict"], **{k: v for k, v in variant.items() if k != "tag"}}
    pred = predict_images(BEST, valid_images, RUN_NAME, "valid", **kwargs)
    pred["split_scheme"] = SPLIT
    pred["git_commit"] = env_info["git_commit"]
    path = save_json(pred, Path("results/predictions") / f"{RUN_NAME}_valid_{variant['tag']}.json", indent=None)
    pred_files.append(path)
    n_boxes = sum(len(e["pred"]) for e in pred["images"])
    med = {k: sorted(e["speed_ms"][k] for e in pred["images"])[len(pred["images"]) // 2]
           for k in pred["images"][0]["speed_ms"]}
    print(f"{variant['tag']:8s}: {len(pred['images'])} images, {n_boxes} boxes, median ms per image {med}")

gt_files = []
for split in ("valid", "test"):   # labels only — no model is run on TEST here
    gt_files.append(save_json(export_ground_truth(EVAL_ROOT / split, CLASS_NAMES, split=split),
                              Path("results/ground_truth") / f"{SPLIT}_{split}.json"))
print("Ground truth written:", [str(p) for p in gt_files])

## 9. Save results (Drive, GitHub results branch, or download)

In [ ]:
import shutil
from porkqc.env import get_secret
from porkqc.colab_utils import push_results

for name in ("results.csv", "args.yaml", "run_meta.json", "results.png"):
    if (RUN_DIR / name).exists():
        shutil.copy2(RUN_DIR / name, OUT / name)
save_json(env_info, OUT / "notebook_env.json")

result_paths = [OUT, *pred_files, *gt_files]
shutil.copytree("results", DRIVE_DIR / "results", dirs_exist_ok=True)
print("Copied to Drive:", DRIVE_DIR / "results")

pushed = False
if PUSH_RESULTS:
    try:
        token = get_secret("GITHUB_TOKEN")
        sha = push_results(REPO_DIR, [str(p) for p in result_paths],
                           f"results: {RUN_NAME} training + VALID predictions (Colab)", token)
        print("Pushed to branch 'colab-results':", sha[:7]); pushed = True
    except (RuntimeError, ValueError) as exc:
        print("Not pushed to GitHub:", exc)
if not pushed:
    archive = shutil.make_archive(f"/content/results_{RUN_NAME}", "zip", ".", "results")
    from google.colab import files
    files.download(archive)
    print("Downloaded", archive, "- attach it in the Claude chat.")

## 10. Hand back to Claude
* If the push worked, just tell Claude "results pushed". Otherwise attach the downloaded zip.
* Optionally *File → Save a copy in GitHub* (path `notebooks/02_train_detector.ipynb`) so the
  executed notebook with its training log is kept.
* Next run: see the run-order table at the top (change `EXPERIMENT` / `SPLIT`).
* **Disconnect the runtime** when done to save GPU quota.